# Эксперимент: целое и порог по разбросу продаж самой пары

Поверх v5. Целое под SMAPE (v4, -0.004...-0.005 score) выбиралось по распределению "если продажа будет" = 1 + Пуассон(q - 1) вокруг прогноза регрессора q. У Пуассона один параметр, поэтому пары с одинаковым q для него неразличимы, и порог по паре (p > 2 / (4 - ожидаемый SMAPE)) выходил почти одинаковым у всех: 0.50-0.56 - и ничего не дал.

Здесь разброс берём из истории самой пары: по ненулевым неделям за 24 недели (`pipeline.pair_dispersion`) оцениваем параметр r отрицательного биномиального (среднее - по-прежнему прогноз регрессора). Стабильная пара "ровно 1 шт." остаётся с Пуассоном; у рваной пары целое ближе к медиане, а порог выше - ошибка при продаже у неё больше. Меньше 4 ненулевых недель или разброс не больше пуассоновского - Пуассон.

Варианты (`dispersion="pair"`): целое по разбросу при общем пороге 0.5 / 0.55; порог по паре (`shift` -0.04 / -0.02 / 0); целое по разбросу с множителем количества 1.1 (целое по разбросу ниже - объём может просесть).

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_disp"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; первый - v5, с ним и сравниваем.

In [2]:
VARIANTS = {
    "v5": V5,
    "целое по разбросу пары": {**V5, "dispersion": "pair"},
    "целое по разбросу пары, порог 0.55": {**V5, "dispersion": "pair", "threshold": 0.55},
    "целое по разбросу пары, множитель 1.1": {**V5, "dispersion": "pair", "amount_scale": 1.1},
}
for s in (-0.04, -0.02, 0.0):
    VARIANTS[f"порог по паре по разбросу, сдвиг {s:+.2f}"] = {**V5, "dispersion": "pair", "shift": s}
list(VARIANTS)

['v5',
 'целое по разбросу пары',
 'целое по разбросу пары, порог 0.55',
 'целое по разбросу пары, множитель 1.1',
 'порог по паре по разбросу, сдвиг -0.04',
 'порог по паре по разбросу, сдвиг -0.02',
 'порог по паре по разбросу, сдвиг +0.00']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

season2025 готов


season2024 готов


recent готов


bias                         rmse  \
фолд                                   recent season2024 season2025 recent   
вариант                                                                      
v5                                      76.51      86.58      81.16   3.34   
порог по паре по разбросу, сдвиг +0.00  74.02      84.82      80.15   3.39   
порог по паре по разбросу, сдвиг -0.02  73.76      84.96      79.78   3.39   
порог по паре по разбросу, сдвиг -0.04  73.84      85.13      79.48   3.38   
целое по разбросу пары                  73.86      84.94      79.68   3.38   
целое по разбросу пары, множитель 1.1   78.75      88.16      82.48   3.25   
целое по разбросу пары, порог 0.55      73.89      85.00      80.36   3.39   

                                                              smape  \
фолд                                   season2024 season2025 recent   
вариант                                                               
v5                                           1.93       2.65  58.71   
порог по паре по разбросу, сдвиг +0.00       1.97       2.69  58.64   
порог по паре по разбросу, сдвиг -0.02       1.97       2.68  58.90   
порог по паре по разбросу, сдвиг -0.04       1.97       2.68  59.16   
целое по разбросу пары                       1.96       2.68  59.10   
целое по разбросу пары, множитель 1.1        1.90       2.63  58.87   
целое по разбросу пары, порог 0.55           1.97       2.69  58.63   

                                                               wape  \
фолд                                   season2024 season2025 recent   
вариант                                                               
v5                                          42.95      57.60  47.07   
порог по паре по разбросу, сдвиг +0.00      43.08      58.02  47.16   
порог по паре по разбросу, сдвиг -0.02      43.16      57.96  47.16   
порог по паре по разбросу, сдвиг -0.04      43.26      58.04  47.18   
целое по разбросу пары                      43.23      58.03  47.18   
целое по разбросу пары, множитель 1.1       43.04      57.79  47.04   
целое по разбросу пары, порог 0.55          43.14      58.15  47.17   

                                                              
фолд                                   season2024 season2025  
вариант                                                       
v5                                          62.96      51.22  
порог по паре по разбросу, сдвиг +0.00      63.13      51.48  
порог по паре по разбросу, сдвиг -0.02      63.11      51.43  
порог по паре по разбросу, сдвиг -0.04      63.14      51.42  
целое по разбросу пары                      63.15      51.43  
целое по разбросу пары, множитель 1.1       63.34      51.63  
целое по разбросу пары, порог 0.55          63.19      51.53

## Выбор

Вариант проходит, если он не хуже v5 (посчитанного в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["v5"]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc["v5", "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print("v5 в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 в этом запуске: {'season2025': 0.8098, 'season2024': 0.8252, 'recent': 0.7867} | по сезону 0.8175
лучший вариант: None - не прошёл ни один


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
v5,0.8098,0.8252,0.7867,0.8175,False
"целое по разбросу пары, множитель 1.1",0.8133,0.8271,0.7864,0.8202,False
"порог по паре по разбросу, сдвиг -0.02",0.8137,0.8289,0.7875,0.8213,False
целое по разбросу пары,0.8139,0.8293,0.7888,0.8216,False
"порог по паре по разбросу, сдвиг +0.00",0.8149,0.8284,0.7862,0.8217,False
"порог по паре по разбросу, сдвиг -0.04",0.8137,0.8298,0.7892,0.8218,False
"целое по разбросу пары, порог 0.55",0.8162,0.8294,0.7862,0.8228,False
